# Balanced CEFR Steering Subset Construction

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Define File Paths
training_csv_path = "/content/drive/Your_Path/efcamdat_classifier_subset.csv"
master_csv_path = "/content/driveYour_Path/efcamdat_master_combined_C2_Cleaned.csv"

print("📥 Loading CSV files...")
training_df = pd.read_csv(training_csv_path)
master_df = pd.read_csv(master_csv_path)

print(f"Training dataset initial shape: {training_df.shape}")
print(f"Master dataset shape: {master_df.shape}")

# 3. Create Lookup Mappings from the Master File
# Map by text_id
text_id_to_topic = master_df.dropna(subset=['text_id', 'topic_title']).set_index('text_id')['topic_title'].to_dict()

# Map by clean_text (fallback)
clean_text_to_topic = master_df.dropna(subset=['clean_text', 'topic_title']).drop_duplicates(subset=['clean_text']).set_index('clean_text')['topic_title'].to_dict()

# 4. Perform Mapping with Fallback
def get_topic(row):
    # Try primary match via text_id
    if pd.notna(row.get('text_id')) and row['text_id'] in text_id_to_topic:
        return text_id_to_topic[row['text_id']]
    # Fallback match via clean_text
    elif pd.notna(row.get('clean_text')) and row['clean_text'] in clean_text_to_topic:
        return clean_text_to_topic[row['clean_text']]
    return None

print("🔍 Searching and matching topic_title from master CSV...")
training_df['topic_title'] = training_df.apply(get_topic, axis=1)

# Report mapping success stats
matched_count = training_df['topic_title'].notna().sum()
missing_count = training_df['topic_title'].isna().sum()
print(f"✅ Successfully matched topic_title for {matched_count} / {len(training_df)} rows.")

if missing_count > 0:
    print(f"⚠️ Warning: {missing_count} rows could not be matched. Assigning default placeholder.")
    training_df['topic_title'] = training_df['topic_title'].fillna("General English Writing Task")

# 5. Overwrite the training CSV in place
training_df.to_csv(training_csv_path, index=False)
print(f"💾 Updated CSV successfully saved back to: {training_csv_path}")

# Display first few rows to verify structure
training_df.head()

📥 Loading CSV files...
Training dataset initial shape: (52657, 4)
Master dataset shape: (620373, 6)
🔍 Searching and matching topic_title from master CSV...
✅ Successfully matched topic_title for 52657 / 52657 rows.
💾 Updated CSV successfully saved back to: /content/drive/MyDrive/Mohammd_Thesis/subsets/efcamdat_classifier_subset.csv


,text_id,level,cefr,clean_text,topic_title
0,EFCAM_618301,10,B2,This morning there was a meeting to decide on ...,Helping a friend find a job
1,EFCAM_419250,7,B1,"Hi, Renee. I was reading the survey results an...",Taking part in a TV viewing survey
2,EFCAM_050464,4,A2,Please mop the floor in the morning everyday. ...,Describing routines
3,EFCAM_190737,1,A1,I never go to small clothing stores. I love go...,Updating your online profile
4,EFCAM_082356,7,B1,When I read the brochure about the cruise to A...,Writing a letter of complaint


In [ ]:
# ==========================================
# 1. DATASET BALANCING ROUTINE
# ==========================================
print("\n⚖️ Checking and Balancing Dataset...")
if not os.path.exists(BALANCED_CSV_PATH):
    print("Balanced dataset not found. Creating it now from the 60K subset...")
    raw_df = pd.read_csv(RAW_CSV_PATH).dropna(subset=['cefr', 'clean_text'])

    # Clean the CEFR column and find the minimum class count (C2)
    raw_df['cefr'] = raw_df['cefr'].str.upper().str.strip()
    c2_count = len(raw_df[raw_df['cefr'] == 'C2'])
    print(f"C2 Baseline Count: {c2_count} samples. Downsampling all classes to match...")

    balanced_dfs = []
    for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
        level_df = raw_df[raw_df['cefr'] == level]
        if len(level_df) > c2_count:
            level_df = level_df.sample(n=c2_count, random_state=42)
        balanced_dfs.append(level_df)

    # Shuffle and save
    balanced_df = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    balanced_df.to_csv(BALANCED_CSV_PATH, index=False)
    print(f"✅ Balanced dataset saved to Drive: {len(balanced_df)} total rows.")
else:
    print("✅ Balanced dataset already exists. Loading directly.")
    balanced_df = pd.read_csv(BALANCED_CSV_PATH).dropna(subset=['cefr', 'clean_text'])

## Balanced CEFR Steering Subset Construction

The Steering Training Dataset is downsampled to obtain an equal number of examples for each of the six CEFR levels. Since C2 is the smallest class with 928 available examples, 928 samples are retained from each level. Sampling is performed with `random_state=42` for reproducibility, resulting in a balanced dataset of 5,568 examples.

In [ ]:
# =========================================================
# BALANCED CEFR STEERING SUBSET CONSTRUCTION
# =========================================================

from pathlib import Path

CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
RANDOM_SEED = 42

# Use the enriched Steering Training Dataset created above.
# It already contains the corresponding topic_title values.
steering_df = training_df.copy()

# Normalize CEFR labels and remove incomplete records.
steering_df["cefr"] = steering_df["cefr"].str.upper().str.strip()
steering_df = steering_df.dropna(subset=["cefr", "clean_text"]).copy()

# ---------------------------------------------------------
# Determine balancing size from the smallest CEFR class
# ---------------------------------------------------------

class_counts = steering_df["cefr"].value_counts()

print("Steering Training Dataset CEFR distribution:")
print(class_counts.reindex(CEFR_LEVELS))

samples_per_level = int(class_counts.reindex(CEFR_LEVELS).min())

print(f"\nSmallest CEFR class: {samples_per_level:,} examples per level")

# The thesis configuration uses 928 examples per CEFR level.
assert samples_per_level == 928, (
    f"Expected 928 examples in the smallest CEFR class, "
    f"but found {samples_per_level}."
)

# ---------------------------------------------------------
# Downsample each CEFR level
# ---------------------------------------------------------

balanced_parts = []

for level in CEFR_LEVELS:
    level_df = steering_df[steering_df["cefr"] == level]

    sampled_df = level_df.sample(
        n=samples_per_level,
        random_state=RANDOM_SEED
    )

    balanced_parts.append(sampled_df)

# Combine the six equally sized subsets and shuffle.
balanced_df = (
    pd.concat(balanced_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

expected_total = samples_per_level * len(CEFR_LEVELS)

assert len(balanced_df) == expected_total
assert balanced_df["text_id"].nunique() == expected_total

balanced_counts = (
    balanced_df["cefr"]
    .value_counts()
    .reindex(CEFR_LEVELS)
)

assert (balanced_counts == samples_per_level).all()

print("\nBalanced CEFR Steering Subset:")
print(balanced_counts)

print(f"\nTotal examples: {len(balanced_df):,}")
print(f"Unique text IDs: {balanced_df['text_id'].nunique():,}")
print(f"Random seed: {RANDOM_SEED}")

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

BALANCED_CSV_PATH = Path(
    "/content/drive/MyDrive/master_thesis_data/subsets/"
    "balanced_cefr_steering_subset.csv"
)

BALANCED_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

balanced_df.to_csv(BALANCED_CSV_PATH, index=False)

print(f"\nBalanced dataset saved to:")
print(BALANCED_CSV_PATH)